In [ ]:
import sqlite3                                 
import pandas as pd                             
 
con = sqlite3.connect('../data/raw/airtel.db')  
 
def q(sql):                                     
    return pd.read_sql(sql, con)

In [ ]:
    q("""SELECT name FROM sqlite_master                  -- SQLite's own list
    WHERE  type = 'table'                           -- tables, not indexes
    ORDER  BY name;""")

In [ ]:
q("""SELECT 'circles' AS tbl, COUNT(*) AS n FROM circles     -- 22 expected
UNION ALL SELECT 'plans',       COUNT(*) FROM plans          -- 14
UNION ALL SELECT 'customers',   COUNT(*) FROM customers      -- 22,000
UNION ALL SELECT 'usage_daily', COUNT(*) FROM usage_daily    -- 1,326,739
UNION ALL SELECT 'recharges',   COUNT(*) FROM recharges      -- 408,089
UNION ALL SELECT 'complaints',  COUNT(*) FROM complaints;    -- 26,841""")

In [ ]:
import sqlite3                                 
import pandas as pd                             
 
pcon = sqlite3.connect('../sql/practice/practice.db')  
pd.read_sql('select * from loans',pcon)
 
def pq(sql):                                     
    return pd.read_sql(sql, pcon)

In [ ]:
pq("""select * from shop_customers;""")

In [ ]:
pq("""select * from shop_orders;""")

In [ ]:
pq("""select * from members;""")

In [ ]:
pq("""select * from loans;""")

In [ ]:
#for each customer ,how many orders did o

pq("""
SELECT
    customer_id,
    COUNT(*) AS total_orders,
    SUM(CASE WHEN amount >= 500 THEN 1 ELSE 0 END) AS orders_500_or_more
FROM shop_orders
GROUP BY customer_id;
""")

In [ ]:
pq(""" select count(*) as orders,
sum(amount>=500) as big_orders,
round(100.0*avg(amount>=500),2)as perc
from shop_orders;""")

In [ ]:
# on 31 march 2025 ,how many days had each customer been with the shop ?
# customer name,joined_date ,no of days 
pq("""
SELECT name,
       joined_on,
       CAST(julianday('2025-03-31') - julianday(joined_on) AS INTEGER) AS no_of_days
FROM shop_customers;
""")


In [ ]:
#how many orders and how much sales in each month?
#month ,order,sales
#shop_orders

pq("""
SELECT strftime('%Y-%m', order_date) AS month,
       COUNT(*) AS orders,
       SUM(amount) AS sales
FROM shop_orders
GROUP BY month
ORDER BY month;
""")

In [ ]:
# which customer have never placed an order ?
pq("""select c.name,c.city 
      from shop_customers as c
      left join shop_orders as o
        on o.customer_id=c.customer_id
     where o.customer_id is null;
     """)

In [ ]:
# which member have never borrowed a book ?
pq("""
select m.name
from members as m 
left join loans as l
   on l.member_id=m.member_id
where l.member_id is null;

""")

In [ ]:
#which customer placed at least one order on after 1st march 2025?
pq("""
 select c.name
 from shop_customers AS c
 where exists (select 1 from shop_orders AS o
               where o.customer_id=c.customer_id
               and o.order_date>='2025-03-01');""")

In [ ]:
pq("""
WITH s AS (
    SELECT customer_id, 
           SUM(amount) AS spent 
    FROM shop_orders 
    GROUP BY customer_id
)
SELECT c.name, 
       s.spent, 
       ROW_NUMBER() OVER (ORDER BY s.spent DESC) AS rn, 
       NTILE(2) OVER (ORDER BY s.spent DESC) AS half
FROM s
JOIN shop_customers c ON s.customer_id = c.customer_id
ORDER BY s.spent DESC;
""")

In [ ]:
def run_sql_file(path):
    with open(path,'r') as f:
        script=f.read()
    pcon.executescript(script)
    pcon.commit()
    print('ran',path)

In [ ]:
run_sql_file('../sql/practice/index.sql')

In [ ]:
pq("""select name,tbl_name from sqlite_master
where type='index';""")

In [ ]:
pq("""select * from shop_orders
where customer_id=2""")

In [ ]:
pq("""
    select julianday('midlage')as id;

""")ant